In [1]:
!pip install -q \
    "sentence-transformers==3.0.1" \
    "transformers==4.46.0" \
    "faiss-cpu==1.8.0" \
    "rank-bm25==0.2.2" \
    "accelerate==0.34.2" \
    "bitsandbytes==0.43.3" \
    "tqdm" "pandas" "numpy" "requests"

!git clone https://github.com/AlibabaResearch/DAMO-ConvAI.git
%cd /content/DAMO-ConvAI/api-bank
!rm -f apis/translate.py
!ls apis/ | head -20

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 96.7 MB/s eta 0:00:00
Cloning into 'DAMO-ConvAI'...
remote: Enumerating objects: 5947, done.
remote: Counting objects: 100% (449/449), done.
remote: Compressing objects: 100% (167/167), done.
remote: Total 5947 (delta 318), reused 282 (delta 282), pack-reused 5498 (from 2)


In [5]:
# CELL 2  —  Imports + paths
import os, sys, json, importlib
import numpy as np
import requests
from tqdm import tqdm
from pathlib import Path

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

APIBANK_REPO  = "/content/DAMO-ConvAI/api-bank"
SEALTOOLS_DIR = "/content/SealTools"
OUTPUT_DIR    = "/content/drive/MyDrive/MCP_Research"

os.makedirs(SEALTOOLS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR,    exist_ok=True)
original_cwd = os.getcwd()
print("✅ Paths ready")
print(f"   OUTPUT_DIR : {OUTPUT_DIR}")

Mounted at /content/drive
✅ Paths ready
   OUTPUT_DIR : /content/drive/MyDrive/MCP_Research


In [6]:
# CELL 3  —  Load API-Bank via ToolManager


def load_apibank_tools() -> list:
    """
    Navigate into api-bank repo and load tools via ToolManager.
    ToolManager imports rank_bm25 internally — we install it first.
    """
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "rank_bm25"],
        check=True
    )

    os.chdir(APIBANK_REPO)
    sys.path.insert(0, APIBANK_REPO)

    # Clear any cached broken imports from before the restart
    for mod in list(sys.modules.keys()):
        if "tool_manager" in mod:
            del sys.modules[mod]

    try:
        from tool_manager import ToolManager
        tm = ToolManager(apis_dir='./apis')
        print(f"✅ ToolManager loaded {len(tm.apis)} tools.")
    except Exception as e:
        os.chdir(original_cwd)
        raise RuntimeError(f"ToolManager failed: {e}")

    tools = []
    for api in tm.apis:
        if isinstance(api, dict):
            tools.append(api)
        else:
            tools.append({
                "name":              getattr(api, "name", type(api).__name__),
                "description":       getattr(api, "description",
                                             (api.__doc__ or "").strip()),
                "input_parameters":  getattr(api, "input_parameters", {}),
                "output_parameters": getattr(api, "output_parameters", {}),
                "domain":            getattr(api, "domain", "general"),
            })

    os.chdir(original_cwd)

    if tools:
        print(f"\n── Sample tool ───────────────────────────────")
        print(f"   name   : {tools[0]['name']}")
        print(f"   desc   : {tools[0]['description'][:80]}")
        print(f"   params : {list(tools[0].get('input_parameters', {}).keys())[:5]}")

    return tools


apibank_raw = load_apibank_tools()
print(f"\nTotal API-Bank tools: {len(apibank_raw)}")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ ToolManager loaded 52 tools.

── Sample tool ───────────────────────────────
   name   : DeleteAccount
   desc   : Delete an account.
   params : ['token']

Total API-Bank tools: 52


In [ ]:
# CELL 4  —  Download + load SealTools


SEALTOOLS_URL  = (
    "https://raw.githubusercontent.com/fairyshine/Seal-Tools/"
    "refs/heads/master/Seal-Tools_Dataset/tool.jsonl"
)
SEALTOOLS_FILE = os.path.join(SEALTOOLS_DIR, "tool.jsonl")


def download_sealtools(url=SEALTOOLS_URL, filepath=SEALTOOLS_FILE):
    print("Downloading SealTools from GitHub...")
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(r.text)
    count = r.text.count("\n")
    print(f"✅ Downloaded ~{count} lines → {filepath}")
    return filepath


def load_jsonl(filepath: str) -> list:
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"  ⚠️ Skipping line {line_num}: {e}")
    return data


if not os.path.exists(SEALTOOLS_FILE):
    download_sealtools()
else:
    print(f"SealTools already at {SEALTOOLS_FILE} — skipping download")

seal_tools_raw = load_jsonl(SEALTOOLS_FILE)
print(f"✅ Loaded {len(seal_tools_raw)} SealTools entries")

# Confirm correct field names
print("\n── First tool keys:", list(seal_tools_raw[0].keys()))
for key in ["api_name", "api_description", "field", "parameters"]:
    count = sum(1 for t in seal_tools_raw if key in t)
    print(f"   '{key}' present in {count}/{len(seal_tools_raw)} tools")

✅ Downloaded ~4076 lines → /content/SealTools/tool.jsonl
✅ Loaded 4076 SealTools entries

── First tool keys: ['api_name', 'api_description', 'field', 'parameters', 'required', 'responses']
   'api_name' present in 4076/4076 tools
   'api_description' present in 4076/4076 tools
   'field' present in 4076/4076 tools
   'parameters' present in 4076/4076 tools


In [ ]:
# CELL 5  —  Convert both datasets to MCP format

TYPE_MAP = {
    "str": "string",    "string": "string",
    "int": "integer",   "integer": "integer",
    "float": "number",  "number": "number",
    "bool": "boolean",  "boolean": "boolean",
    "list": "array",    "list(str)": "array",
    "dict": "object",   "json": "object",   "object": "object",
    "any":  "string",
}

def map_type(t: str) -> str:
    return TYPE_MAP.get(str(t).strip().lower(), "string")


def convert_apibank_to_mcp(api: dict) -> dict:
    properties = {}
    required   = []
    for pname, pinfo in api.get("input_parameters", {}).items():
        if isinstance(pinfo, dict):
            raw_t = pinfo.get("type", "str")
            prop  = {
                "type":        map_type(raw_t),
                "description": pinfo.get("description", ""),
            }
            if map_type(raw_t) == "array":
                prop["items"] = {"type": "string"}
            properties[pname] = prop
        else:
            properties[pname] = {"type": map_type(str(pinfo)), "description": ""}
        required.append(pname)

    name   = api.get("name", "")
    desc   = api.get("description", "")
    domain = api.get("domain", "general")

    mcp = {
        "name":        name,
        "description": desc,
        "domain":      domain,
        "source":      "api_bank",
        "inputSchema": {
            "type": "object", "properties": properties, "required": required
        },
    }

    out_props = {}
    for pname, pinfo in api.get("output_parameters", {}).items():
        if isinstance(pinfo, dict):
            out_props[pname] = {
                "type":        map_type(pinfo.get("type", "str")),
                "description": pinfo.get("description", ""),
            }
    if out_props:
        mcp["outputSchema"] = {
            "type": "object", "properties": out_props,
            "required": list(out_props)
        }
    return mcp


def convert_sealtools_to_mcp(tool: dict) -> dict:
    name = tool.get("api_name") or "unknown_tool"
    desc = tool.get("api_description") or "No description provided"
    raw_field = tool.get("field") or "general"
    if "/" in raw_field:
        raw_field = raw_field.split("/")[-1].strip()
    domain = raw_field.lower().replace(" ", "_").replace("-", "_")

    parameters = tool.get("parameters") or {}
    top_required = tool.get("required") or []

    properties = {}
    for pname, pinfo in parameters.items():
        if isinstance(pinfo, dict):
            raw_t = pinfo.get("type", "string")
            pdesc = pinfo.get("description", "")
            prop = {"type": map_type(raw_t), "description": pdesc}
            if map_type(raw_t) == "array":
                prop["items"] = {"type": "string"}
            properties[pname] = prop
        else:
            properties[pname] = {"type": "string", "description": str(pinfo) if pinfo else ""}

    mcp = {
        "name": name,
        "description": desc,
        "domain": domain,
        "source": "seal_tools",
        "inputSchema": {
            "type": "object",
            "properties": properties,
            "required": [p for p in top_required if p in properties]  # only valid ones
        },
    }

    # Fixed: Handle SealTools "responses" field
    responses = tool.get("responses")
    if responses and isinstance(responses, dict):
        out_props = {}
        for rname, rinfo in responses.items():
            if isinstance(rinfo, dict):
                out_props[rname] = {
                    "type": map_type(rinfo.get("type", "string")),
                    "description": rinfo.get("description", ""),
                }
        if out_props:
            mcp["outputSchema"] = {
                "type": "object",
                "properties": out_props,
                "required": list(out_props.keys())
            }

    return mcp


# Run conversions
print("Converting API-Bank → MCP...")
apibank_mcp = []
for api in tqdm(apibank_raw, desc="API-Bank"):
    try:
        apibank_mcp.append(convert_apibank_to_mcp(api))
    except Exception as e:
        print(f"  ⚠️  {api.get('name','?')}: {e}")

print(f"✅ API-Bank  : {len(apibank_mcp)} tools")

print("\nConverting SealTools → MCP...")
sealtools_mcp = []
seal_errors   = []
for idx, tool in enumerate(tqdm(seal_tools_raw, desc="SealTools")):
    try:
        sealtools_mcp.append(convert_sealtools_to_mcp(tool))
    except Exception as e:
        seal_errors.append((idx, tool.get("api_name", "?"), str(e)))

print(f"✅ SealTools : {len(sealtools_mcp)} tools")
if seal_errors:
    print(f"⚠️  Errors   : {len(seal_errors)}")
    for idx, name, err in seal_errors[:3]:
        print(f"   [{idx}] {name}: {err}")


# Domain stats
print("\n── API-Bank domains:")
ab_dom = {}
for t in apibank_mcp:
    ab_dom[t["domain"]] = ab_dom.get(t["domain"], 0) + 1
for d, c in sorted(ab_dom.items(), key=lambda x: -x[1]):
    print(f"   {d:40s} {c}")

print("\n── SealTools top-10 domains:")
st_dom = {}
for t in sealtools_mcp:
    st_dom[t["domain"]] = st_dom.get(t["domain"], 0) + 1
for d, c in sorted(st_dom.items(), key=lambda x: -x[1])[:10]:
    print(f"   {d:40s} {c}")


Converting SealTools → MCP...


SealTools: 100%|██████████| 4076/4076 [00:00<00:00, 161334.61it/s]

✅ SealTools : 4076 tools

── SealTools top-10 domains:
   robotics                                 17
   frontend_development                     12
   geology                                  11
   biotechnology                            11
   vehicle_dynamics                         11
   cybersecurity                            11
   database_management                      11
   biometrics                               11
   fpga_design                              11
   quality_assurance                        10


In [ ]:
# CELL 5 — Convert SealTools only to MCP format (final version - duplicate of above with better error handling and domain extraction)

from tqdm import tqdm

TYPE_MAP = {
    "str": "string",
    "string": "string",
    "int": "integer",
    "integer": "integer",
    "float": "number",
    "number": "number",
    "bool": "boolean",
    "boolean": "boolean",
    "list": "array",
    "list(str)": "array",
    "dict": "object",
    "json": "object",
    "object": "object",
    "any": "string",
}

def map_type(t) -> str:
    return TYPE_MAP.get(str(t).strip().lower(), "string")


def make_schema_from_parameters(parameters: dict, required_fields=None) -> dict:
    if required_fields is None:
        required_fields = []

    properties = {}

    for pname, pinfo in (parameters or {}).items():
        if isinstance(pinfo, dict):
            raw_t = pinfo.get("type", "string")
            desc = pinfo.get("description", "")

            prop = {
                "type": map_type(raw_t),
                "description": desc,
            }

            if prop["type"] == "array":
                prop["items"] = {"type": "string"}

            properties[pname] = prop
        else:
            raw_t = str(pinfo) if pinfo is not None else "string"
            prop = {
                "type": map_type(raw_t),
                "description": "",
            }
            if prop["type"] == "array":
                prop["items"] = {"type": "string"}
            properties[pname] = prop

    return {
        "type": "object",
        "properties": properties,
        "required": [p for p in required_fields if p in properties]
    }


def make_output_schema_from_responses(responses: dict):
    if not responses or not isinstance(responses, dict):
        return None

    out_props = {}

    for rname, rinfo in responses.items():
        if isinstance(rinfo, dict):
            raw_t = rinfo.get("type", "string")
            desc = rinfo.get("description", "")
            prop = {
                "type": map_type(raw_t),
                "description": desc,
            }
            if prop["type"] == "array":
                prop["items"] = {"type": "string"}
            out_props[rname] = prop
        else:
            out_props[rname] = {
                "type": map_type(str(rinfo)),
                "description": "",
            }

    if not out_props:
        return None

    return {
        "type": "object",
        "properties": out_props,
        "required": list(out_props.keys())
    }


# ← REMOVED: duplicate first definition of convert_sealtools_to_strict_mcp


def convert_sealtools_to_strict_mcp(tool: dict) -> dict:
    name = tool.get("api_name") or "unknown_tool"
    desc = tool.get("api_description") or "No description provided"
    domain = tool.get("field") or "general"          # ← ADDED

    input_schema = make_schema_from_parameters(
        parameters=tool.get("parameters", {}),
        required_fields=tool.get("required", [])
    )

    tool_obj = {
        "name": name,
        "description": desc,
        "domain": domain,                             # ← ADDED
        "inputSchema": input_schema,
    }
    output_schema = make_output_schema_from_responses(tool.get("responses", {}))
    if output_schema:
        tool_obj["outputSchema"] = output_schema

    return tool_obj


def convert_sealtools_dataset(seal_tools_raw, mode="strict_mcp"):
    if mode not in {"strict_mcp", "mcp_with_metadata"}:
        raise ValueError("mode must be 'strict_mcp' or 'mcp_with_metadata'")

    print(f"Running conversion mode: {mode}")
    print("\nConverting SealTools...")

    sealtools_mcp = []
    seal_errors = []

    for idx, tool in enumerate(tqdm(seal_tools_raw, desc="SealTools")):
        try:
            if mode == "strict_mcp":
                sealtools_mcp.append(convert_sealtools_to_strict_mcp(tool))
            else:
                sealtools_mcp.append(convert_sealtools_to_mcp_with_metadata(tool))
        except Exception as e:
            seal_errors.append((idx, tool.get("api_name", "?"), str(e)))

    print(f"\n✅ SealTools converted: {len(sealtools_mcp)}")

    if seal_errors:
        print(f"\n⚠️ SealTools errors: {len(seal_errors)}")
        for idx, name, err in seal_errors[:5]:
            print(f"   [{idx}] {name}: {err}")

    return sealtools_mcp, seal_errors


#RUN

CONVERSION_MODE = "strict_mcp"
# CONVERSION_MODE = "mcp_with_metadata"

sealtools_mcp, seal_errors = convert_sealtools_dataset(
    seal_tools_raw=seal_tools_raw,
    mode=CONVERSION_MODE
)

print(f"\nConverted SealTools: {len(sealtools_mcp)}")

if sealtools_mcp:
    print("\nSample SealTools MCP tool:")
    print(sealtools_mcp[0])

Running conversion mode: strict_mcp

Converting SealTools...


SealTools: 100%|██████████| 4076/4076 [00:00<00:00, 37685.98it/s]


✅ SealTools converted: 4076

Converted SealTools: 4076

Sample SealTools MCP tool:
{'name': 'analyzeEvidence', 'description': 'Analyze the chemical evidence collected from a crime scene', 'domain': 'Chemical Engineering/Forensic engineering', 'inputSchema': {'type': 'object', 'properties': {'evidence_type': {'type': 'string', 'description': 'The type of evidence to be analyzed (e.g., DNA, fingerprints, blood, fibers)'}, 'method': {'type': 'string', 'description': 'The method or technique to be used for analysis (e.g., spectroscopy, chromatography, microscopy)'}, 'sample': {'type': 'string', 'description': 'The sample or specimen to be analyzed (e.g., crime scene swab, hair strand, fabric sample)'}}, 'required': ['evidence_type', 'method', 'sample']}, 'outputSchema': {'type': 'object', 'properties': {'analysis_results': {'type': 'string', 'description': 'The results of the chemical analysis of the evidence'}, 'conclusion': {'type': 'string', 'description': 'The conclusion drawn from th

In [ ]:
sample = seal_tools_raw[0]
mcp_tool = convert_sealtools_to_strict_mcp(sample)
print("Input required:", mcp_tool["inputSchema"]["required"])
print("Output schema:", mcp_tool.get("outputSchema"))

Input required: ['evidence_type', 'method', 'sample']
Output schema: {'type': 'object', 'properties': {'analysis_results': {'type': 'string', 'description': 'The results of the chemical analysis of the evidence'}, 'conclusion': {'type': 'string', 'description': 'The conclusion drawn from the analysis'}}, 'required': ['analysis_results', 'conclusion']}


In [ ]:
import os
import json

reg_fname = "sealtools_mcp_registry.json"
reg_path = os.path.join(OUTPUT_DIR, reg_fname)
with open(reg_path, "w") as f:
    json.dump({"tools": sealtools_mcp, "total_tools": len(sealtools_mcp)}, f, indent=2)

kb = os.path.getsize(reg_path) / 1024
print(f"   ✅ {reg_fname:45s} ({len(sealtools_mcp):5d} tools,  {kb:7.1f} KB)")

   ✅ sealtools_mcp_registry.json                   ( 4076 tools,   4345.3 KB)


In [ ]:
# CELL 6  —  Embed + save to Drive

from sentence_transformers import SentenceTransformer

EMBED_MODEL = "all-MiniLM-L6-v2"
print(f"Loading embedding model: {EMBED_MODEL}")
embedder = SentenceTransformer(EMBED_MODEL)
print("✅ Model loaded")


def embed_tools(mcp_tools: list, embedder, batch_size=64) -> np.ndarray:
    texts = [
        f"Tool: {t['name']} | Description: {t['description']}"
        for t in mcp_tools
    ]
    embs = embedder.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")
    print(f"   shape: {embs.shape}")
    return embs


print("\nEmbedding API-Bank tools...")
apibank_embs = embed_tools(apibank_mcp, embedder)

print("\nEmbedding SealTools tools...")
sealtools_embs = embed_tools(sealtools_mcp, embedder)

# Combined registry (deduplicated by name)
seen, combined = set(), []
for t in apibank_mcp + sealtools_mcp:
    if t["name"] not in seen:
        seen.add(t["name"])
        combined.append(t)
print(f"\n✅ Combined registry: {len(combined)} unique tools")

print("\nEmbedding combined tools...")
combined_embs = embed_tools(combined, embedder)

# Save to Drive
print(f"\nSaving to {OUTPUT_DIR} ...")
save_items = [
    (apibank_mcp,   apibank_embs,   "api_bank_mcp_registry.json",  "api_bank_embeddings.npy"),
    (sealtools_mcp, sealtools_embs, "sealtools_mcp_registry.json", "sealtools_embeddings.npy"),
    (combined,      combined_embs,  "combined_mcp_registry.json",  "combined_embeddings.npy"),
]

for tools_list, embs, reg_fname, emb_fname in save_items:
    reg_path = os.path.join(OUTPUT_DIR, reg_fname)
    with open(reg_path, "w") as f:
        json.dump({"tools": tools_list, "total_tools": len(tools_list)}, f, indent=2)

    emb_path = os.path.join(OUTPUT_DIR, emb_fname)
    np.save(emb_path, embs)

    kb = os.path.getsize(reg_path) / 1024
    print(f"   ✅ {reg_fname:45s} ({len(tools_list):5d} tools,  {kb:7.1f} KB)")
    print(f"   ✅ {emb_fname:45s} shape={embs.shape}")



Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

   shape: (52, 384)

Embedding SealTools tools...


Batches:   0%|          | 0/64 [00:00<?, ?it/s]

   shape: (4076, 384)

✅ Combined registry: 4128 unique tools

Embedding combined tools...


Batches:   0%|          | 0/65 [00:00<?, ?it/s]

   shape: (4128, 384)

Saving to /content/drive/MyDrive/MCP_Research ...
   ✅ api_bank_mcp_registry.json                    (   52 tools,     50.8 KB)
   ✅ api_bank_embeddings.npy                       shape=(52, 384)
   ✅ sealtools_mcp_registry.json                   ( 4076 tools,   2691.5 KB)
   ✅ sealtools_embeddings.npy                      shape=(4076, 384)
   ✅ combined_mcp_registry.json                    ( 4128 tools,   2742.3 KB)
   ✅ combined_embeddings.npy                       shape=(4128, 384)
